In [1]:
##modules
#%matplotlib widget
#%matplotlib inline
#
%matplotlib qt
import mne
import numpy as np
import matplotlib
# Establecer un backend interactivo, como 'Qt5Agg', 'GTK3Agg', etc.
# Esto depende de los backends disponibles en tu sistema.

import matplotlib.pyplot as plt

#matplotlib.use('TkAgg')  # Asegúrate de que este backend está instalado.

import pandas as pd 
import os
import sys

from mne.preprocessing import ICA, corrmap, create_ecg_epochs, create_eog_epochs

from os.path import join as pathjoin
from time import time

from pathlib import Path

from autoreject import AutoReject

# aplicar la acf EN epochs

from statsmodels.tsa.stattools import acf
import numpy as np

import pandas as pd


import os, re, warnings
import numpy as np
import pandas as pd
import mne
from scipy.sparse import csr_matrix


import pickle
from scipy.sparse import triu

from mne.channels.layout import _find_topomap_coords

In [11]:

try:
    # Si se ejecuta como SCRIPT .py: usar __file__
    sys.path.append(os.path.abspath(os.path.join(os.path.dirname(__file__), "..")))
except NameError:
    # Si se ejecuta como NOTEBOOK Jupyter: usar path relativo
    sys.path.append("..")  # sube un nivel desde la carpeta actual del notebook


# --- Configuración dinámica de rutas ---
from get_paths_SELF import get_paths_SELF

# Parámetros editables
disco = "g"
modality="visual"
layer_script = "event"
subj= "s01b"
type_epoch = "emoc"


# Generar variables automáticamente
path_dict = get_paths_SELF(disco=disco, modality=modality,layer_script=layer_script,  subj=subj)
globals().update(path_dict)

# Mostrar todos los paths generados
print("\n📁 Rutas generadas:")
for k, v in path_dict.items():
    print(f"{k:<20} → {v}")
    
    
    

#modify this only if you want to use dynamic analysis, or static event analysis
if layer_script=="event":
    dynamic=False
    
elif layer_script=="block":
    dynamic=False # it will ALWAYS be false in block



    

✅ Carpeta creada: g:\PROYECTO_SELF\SELF_visual\output_preproc\preproc_event\epochs_event
✅ Carpeta creada: g:\PROYECTO_SELF\SELF_visual\output_preproc\preproc_event\ICA_event
✅ Carpeta creada: g:\PROYECTO_SELF\SELF_visual\output_preproc\preproc_event\epochs_clean_event
✅ Carpeta creada: g:\PROYECTO_SELF\SELF_visual\output_preproc\preproc_event\epochs_matlab_event
✅ Carpeta creada: g:\PROYECTO_SELF\SELF_visual\output_preproc\preproc_event\evoked_event
✅ Carpeta creada: g:\PROYECTO_SELF\SELF_visual\channels_structure
✅ Carpeta creada: g:\PROYECTO_SELF\SELF_visual\output_source\source_event
✅ Carpeta creada: g:\PROYECTO_SELF\SELF_visual\output_source\source_event\raw_hsp
✅ Carpeta creada: g:\PROYECTO_SELF\SELF_visual\output_source\source_event\fwd
✅ Carpeta creada: g:\PROYECTO_SELF\SELF_visual\output_source\source_event\inverse
✅ Carpeta creada: g:\PROYECTO_SELF\SELF_visual\output_analysis\analysis_event\acw_event
✅ Carpeta creada: g:\PROYECTO_SELF\SELF_visual\output_analysis\analysis_eve

In [ ]:

# ------------------------------------------------------------
# Parameters
# ------------------------------------------------------------

variables = ["ACW_0", "ACW_50"]

effects_to_plot = [
    "Condition_self",
    "Condition_emotion",
    "Condition_self:Condition_emotion"
]


models_dir = Path(r"G:\PROYECTO_SELF\models output")

# ------------------------------------------------------------
# Load raw MEG/EEG file only for sensor positions
# ------------------------------------------------------------
# OJO:
# Si ahora estás en EEG, cambia esta parte para leer epochs/info EEG.
# Este bloque es el de MEG/CTF que ya tenías.

fname = epochs_clean_path / f"{subj}_epochs_{type_epoch}-epo.fif"



# Cargar epochs del sujeto
epochs = mne.read_epochs(fname, preload=True)

# ------------------------------------------------------------
# Get 2D sensor coordinates once
# ------------------------------------------------------------

picks = mne.pick_types(epochs.info, eeg=True, exclude=[])
pos = _find_topomap_coords(epochs.info, picks=picks)
raw_ch_names = [epochs.ch_names[p] for p in picks]


# ------------------------------------------------------------
# Helper function
# ------------------------------------------------------------

def plot_cluster_channels(
    cluster_channels,
    raw_ch_names,
    pos,
    dv,
    effect_name,
    cluster_id,
    layer_script,
    type_epoch,
):
    cluster_channels = set(map(str, cluster_channels))

    xs_cluster, ys_cluster = [], []
    xs_other, ys_other = [], []

    for ch, xy in zip(raw_ch_names, pos):

        # Para CTF/MEG mantienes sufijo -4304.
        # Para EEG probablemente NO quieras añadir sufijo.
        ch_suf = ch if ch.endswith("-4304") else f"{ch}-4304"

        if ch_suf in cluster_channels or ch in cluster_channels:
            xs_cluster.append(xy[0])
            ys_cluster.append(xy[1])
        else:
            xs_other.append(xy[0])
            ys_other.append(xy[1])

    print(
        f"{dv} | {effect_name} | cluster {cluster_id} | "
        f"plotted cluster channels: {len(xs_cluster)}"
    )

    plt.figure(figsize=(7, 7))

    plt.scatter(
        xs_other,
        ys_other,
        s=25,
        c="lightgrey",
        edgecolors="black",
        linewidths=0.3,
        label="Other channels"
    )

    plt.scatter(
        xs_cluster,
        ys_cluster,
        s=80,
        c="red",
        edgecolors="black",
        linewidths=0.8,
        label=f"Cluster {cluster_id}"
    )

    plt.title(
        f"{dv} | {effect_name}\n"
        f"Cluster {cluster_id} | {layer_script} | {type_epoch}"
    )

    plt.axis("equal")
    plt.axis("off")
    plt.legend()
    plt.show()


# ------------------------------------------------------------
# Loop over DVs, effects, and significant clusters
# ------------------------------------------------------------

for dv in variables:

    for effect_name in effects_to_plot:

        effect_safe = effect_name.replace(":", "_x_")

        output_suffix = (
            f"_{dv}_{effect_safe}_{layer_script}"
            + ("_dynamic" if dynamic else "")
            + f"_{type_epoch}"
        )

        cluster_csv = models_dir / f"significant_cluster_channels{output_suffix}.csv"

        print("\n============================================================")
        print("DV:", dv)
        print("Effect:", effect_name)
        print("CSV:", cluster_csv)
        print("============================================================")

        if not cluster_csv.exists():
            print("⚠️ File not found, skipping.")
            continue

        cluster_df = pd.read_csv(cluster_csv)

        if cluster_df.empty:
            print("No significant clusters in this file.")
            continue

        for cluster_id_target in sorted(cluster_df["cluster_id"].unique()):

            cluster_channels = (
                cluster_df.loc[
                    cluster_df["cluster_id"] == cluster_id_target,
                    "Channel"
                ]
                .astype(str)
                .tolist()
            )

            print("N cluster channels:", len(cluster_channels))
            print(cluster_channels[:10])

            plot_cluster_channels(
                cluster_channels=cluster_channels,
                raw_ch_names=raw_ch_names,
                pos=pos,
                dv=dv,
                effect_name=effect_name,
                cluster_id=cluster_id_target,
                layer_script=layer_script,
                type_epoch=type_epoch,
            )

Reading g:\PROYECTO_SELF\SELF_visual\output_preproc\preproc_event\epochs_clean_event\s01b_epochs_emoc-epo.fif ...


    Found the data of interest:
        t =       0.00 ...    6000.00 ms
        0 CTF compensation matrices available
Not setting metadata
212 matching events found
No baseline correction applied
0 projection items activated

DV: ACW_0
Effect: Condition_self
CSV: G:\PROYECTO_SELF\models output\significant_cluster_channels_ACW_0_Condition_self_event_emoc.csv
No significant clusters in this file.

DV: ACW_0
Effect: Condition_emotion
CSV: G:\PROYECTO_SELF\models output\significant_cluster_channels_ACW_0_Condition_emotion_event_emoc.csv
N cluster channels: 36
['F5', 'F3', 'F1', 'Fz', 'F2', 'F6', 'F8', 'FT7', 'FC5', 'FC3']
ACW_0 | Condition_emotion | cluster 3 | plotted cluster channels: 36

DV: ACW_0
Effect: Condition_self:Condition_emotion
CSV: G:\PROYECTO_SELF\models output\significant_cluster_channels_ACW_0_Condition_self_x_Condition_emotion_event_emoc.csv
No significant clusters in this file.

DV: ACW_50
Effect: Condition_self
CSV: G:\PROYECTO_SELF\models output\significant_cluster_ch

In [4]:
# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------

channels_structure_path = Path(r"G:\MOUS_204\channels_structure")
models_output_path = Path(r"G:\MOUS_204\models output")

adjacency_csv_path = channels_structure_path / "adjacency_reduced_visual.csv"
cluster_csv_path = models_output_path / "ACW0_condition_significant_cluster_channels.csv"

cluster_id_target = 1

# ------------------------------------------------------------
# Load reduced adjacency directly from CSV
# ------------------------------------------------------------

adjacency_df = pd.read_csv(adjacency_csv_path, index_col=0)

channel_order = adjacency_df.index.astype(str).tolist()

print("Adjacency shape:", adjacency_df.shape)
print("N channels in adjacency:", len(channel_order))

assert adjacency_df.shape[0] == adjacency_df.shape[1]
assert list(adjacency_df.index.astype(str)) == list(adjacency_df.columns.astype(str))

# ------------------------------------------------------------
# Load significant cluster channels
# ------------------------------------------------------------

cluster_df = pd.read_csv(cluster_csv_path)

cluster_channels = (
    cluster_df.loc[
        cluster_df["cluster_id"] == cluster_id_target,
        "Channel"
    ]
    .astype(str)
    .tolist()
)

cluster_channels = set(cluster_channels)

print("N cluster channels from CSV:", len(cluster_channels))

missing_from_adjacency = cluster_channels - set(channel_order)

print("N cluster channels missing from adjacency:", len(missing_from_adjacency))
print(sorted(missing_from_adjacency))

# ------------------------------------------------------------
# Load raw MEG file only for sensor positions
# ------------------------------------------------------------

raw = mne.io.read_raw_ctf(
    pathjoin(meg_dir, f"{subj}_task-visual_meg.ds"),
    preload=False
)

raw_meg = raw.copy().pick(picks="mag", exclude=[])

picks = mne.pick_types(raw_meg.info, meg="mag", exclude=[])
pos = _find_topomap_coords(raw_meg.info, picks=picks)
raw_ch_names = [raw_meg.ch_names[p] for p in picks]

# ------------------------------------------------------------
# Build position map
# ------------------------------------------------------------

pos_map = {}

for ch, xy in zip(raw_ch_names, pos):
    ch_suf = ch if ch.endswith("-4304") else f"{ch}-4304"
    pos_map[ch_suf] = xy

missing_positions = sorted(set(channel_order) - set(pos_map))

print("N adjacency channels missing positions in this raw:", len(missing_positions))
print(missing_positions[:20])

# ------------------------------------------------------------
# Build edge list directly from adjacency dataframe
# ------------------------------------------------------------

adjacency_array = adjacency_df.to_numpy()

rows, cols = np.where(np.triu(adjacency_array, k=1) != 0)

edges = pd.DataFrame({
    "from_channel": [channel_order[i] for i in rows],
    "to_channel": [channel_order[j] for j in cols],
})

print("N total adjacency edges:", len(edges))

# ------------------------------------------------------------
# Keep only edges inside the selected cluster
# ------------------------------------------------------------

cluster_edges = edges[
    edges["from_channel"].isin(cluster_channels) &
    edges["to_channel"].isin(cluster_channels)
].copy()

print("N cluster internal edges:", len(cluster_edges))

# ------------------------------------------------------------
# Check if cluster channels have internal edges
# ------------------------------------------------------------

connected_cluster_channels = (
    set(cluster_edges["from_channel"]) |
    set(cluster_edges["to_channel"])
)

isolated_cluster_channels = sorted(cluster_channels - connected_cluster_channels)

print("N isolated cluster channels according to adjacency:", len(isolated_cluster_channels))
print(isolated_cluster_channels)

# ------------------------------------------------------------
# Prepare plot coordinates
# ------------------------------------------------------------

xs_cluster, ys_cluster = [], []
xs_other, ys_other = [], []

for ch in channel_order:
    if ch not in pos_map:
        continue

    x, y = pos_map[ch]

    if ch in cluster_channels:
        xs_cluster.append(x)
        ys_cluster.append(y)
    else:
        xs_other.append(x)
        ys_other.append(y)

print("Plotted cluster channels:", len(xs_cluster))
print("Plotted other channels:", len(xs_other))

# ------------------------------------------------------------
# Plot with real adjacency edges inside cluster
# ------------------------------------------------------------

plt.figure(figsize=(8, 8))

plt.scatter(
    xs_other,
    ys_other,
    s=20,
    c="lightgrey",
    edgecolors="black",
    linewidths=0.3,
    label="Other channels"
)

for _, row in cluster_edges.iterrows():
    ch_a = row["from_channel"]
    ch_b = row["to_channel"]

    if ch_a in pos_map and ch_b in pos_map:
        xa, ya = pos_map[ch_a]
        xb, yb = pos_map[ch_b]

        plt.plot(
            [xa, xb],
            [ya, yb],
            c="red",
            linewidth=0.7,
            alpha=0.45
        )

plt.scatter(
    xs_cluster,
    ys_cluster,
    s=80,
    c="red",
    edgecolors="black",
    linewidths=0.8,
    label=f"Cluster {cluster_id_target}"
)

plt.title(f"ACW_0 Condition cluster {cluster_id_target} with adjacency edges")
plt.axis("equal")
plt.axis("off")
plt.legend()
plt.show()

Adjacency shape: (270, 270)
N channels in adjacency: 270
N cluster channels from CSV: 187
N cluster channels missing from adjacency: 0
[]
ds directory : g:\MOUS_204\sub-V1001\meg\sub-V1001_task-visual_meg.ds
    res4 data read.
    hc data read.
    Separate EEG position data file not present.
    Quaternion matching (desired vs. transformed):
      -0.50   80.15    0.00 mm <->   -0.50   80.15    0.00 mm (orig :  -71.73   42.77 -259.14 mm) diff =    0.000 mm
       0.50  -80.15    0.00 mm <->    0.50  -80.15    0.00 mm (orig :   37.79  -74.15 -264.78 mm) diff =    0.000 mm
     108.63    0.00    0.00 mm <->  108.63   -0.00    0.00 mm (orig :   62.71   58.11 -264.02 mm) diff =    0.000 mm
    Coordinate transformations established.
    Polhemus data for 3 HPI coils added
    Device coordinate locations for 3 HPI coils added
Picked positions of 4 EEG channels from channel info
    4 EEG locations added to Polhemus data.
    Measurement info composed.
Finding samples for g:\MOUS_204\sub-V

In [5]:
from pathlib import Path
from os.path import join as pathjoin

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import mne
from mne.channels.layout import _find_topomap_coords

# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------

channels_structure_path = Path(r"G:\MOUS_204\channels_structure")
models_output_path = Path(r"G:\MOUS_204\models output")

adjacency_csv_path = channels_structure_path / "adjacency_reduced_visual.csv"
cluster_csv_path = models_output_path / "ACW0_condition_significant_cluster_channels.csv"

cluster_id_target = 1

# ------------------------------------------------------------
# Load adjacency directly from CSV
# ------------------------------------------------------------

adjacency_df = pd.read_csv(adjacency_csv_path, index_col=0)
channel_order = adjacency_df.index.astype(str).tolist()

# ------------------------------------------------------------
# Load cluster channels
# ------------------------------------------------------------

cluster_df = pd.read_csv(cluster_csv_path)

cluster_channels = (
    cluster_df.loc[
        cluster_df["cluster_id"] == cluster_id_target,
        "Channel"
    ]
    .astype(str)
    .tolist()
)

cluster_channels = set(cluster_channels)

print("N cluster channels:", len(cluster_channels))

# ------------------------------------------------------------
# Load raw only for sensor positions
# ------------------------------------------------------------

raw = mne.io.read_raw_ctf(
    pathjoin(meg_dir, f"{subj}_task-visual_meg.ds"),
    preload=False
)

raw_meg = raw.copy().pick(picks="mag", exclude=[])

picks = mne.pick_types(raw_meg.info, meg="mag", exclude=[])
pos = _find_topomap_coords(raw_meg.info, picks=picks)
raw_ch_names = [raw_meg.ch_names[p] for p in picks]

# ------------------------------------------------------------
# Build position map
# ------------------------------------------------------------

pos_map = {}
for ch, xy in zip(raw_ch_names, pos):
    ch_suf = ch if ch.endswith("-4304") else f"{ch}-4304"
    pos_map[ch_suf] = xy

# ------------------------------------------------------------
# Build edge list from adjacency matrix
# ------------------------------------------------------------

adjacency_array = adjacency_df.to_numpy()

rows, cols = np.where(np.triu(adjacency_array, k=1) != 0)

edges = pd.DataFrame({
    "from_channel": [channel_order[i] for i in rows],
    "to_channel": [channel_order[j] for j in cols],
})

# Only edges fully inside selected cluster
cluster_edges = edges[
    edges["from_channel"].isin(cluster_channels) &
    edges["to_channel"].isin(cluster_channels)
].copy()

print("N cluster internal edges:", len(cluster_edges))

# ------------------------------------------------------------
# Split channels for plotting
# ------------------------------------------------------------

xs_cluster, ys_cluster, labels_cluster = [], [], []
xs_other, ys_other = [], []

for ch in channel_order:
    if ch not in pos_map:
        continue

    x, y = pos_map[ch]

    if ch in cluster_channels:
        xs_cluster.append(x)
        ys_cluster.append(y)
        labels_cluster.append(ch)
    else:
        xs_other.append(x)
        ys_other.append(y)

# ------------------------------------------------------------
# Plot
# ------------------------------------------------------------

plt.figure(figsize=(14, 14))

# Other channels
plt.scatter(
    xs_other,
    ys_other,
    s=25,
    c="lightgrey",
    edgecolors="black",
    linewidths=0.3,
    label="Other channels"
)

# Cluster edges
for _, row in cluster_edges.iterrows():
    ch_a = row["from_channel"]
    ch_b = row["to_channel"]

    if ch_a in pos_map and ch_b in pos_map:
        xa, ya = pos_map[ch_a]
        xb, yb = pos_map[ch_b]

        plt.plot(
            [xa, xb],
            [ya, yb],
            c="red",
            linewidth=1.2,
            alpha=0.55
        )

# Cluster channels
plt.scatter(
    xs_cluster,
    ys_cluster,
    s=120,
    c="red",
    edgecolors="black",
    linewidths=0.9,
    label=f"Cluster {cluster_id_target}"
)

# Big labels for cluster channels
for x, y, label in zip(xs_cluster, ys_cluster, labels_cluster):
    plt.text(
        x, y, label.replace("-4304", ""),
        fontsize=10,
        fontweight="bold",
        ha="left",
        va="bottom",
        color="black"
    )

plt.title(f"ACW_0 Condition cluster {cluster_id_target} with channel names", fontsize=16)
plt.axis("equal")
plt.axis("off")
plt.legend(fontsize=12)
plt.show()

N cluster channels: 187
ds directory : g:\MOUS_204\sub-V1001\meg\sub-V1001_task-visual_meg.ds
    res4 data read.
    hc data read.
    Separate EEG position data file not present.
    Quaternion matching (desired vs. transformed):
      -0.50   80.15    0.00 mm <->   -0.50   80.15    0.00 mm (orig :  -71.73   42.77 -259.14 mm) diff =    0.000 mm
       0.50  -80.15    0.00 mm <->    0.50  -80.15    0.00 mm (orig :   37.79  -74.15 -264.78 mm) diff =    0.000 mm
     108.63    0.00    0.00 mm <->  108.63   -0.00    0.00 mm (orig :   62.71   58.11 -264.02 mm) diff =    0.000 mm
    Coordinate transformations established.
    Polhemus data for 3 HPI coils added
    Device coordinate locations for 3 HPI coils added
Picked positions of 4 EEG channels from channel info
    4 EEG locations added to Polhemus data.
    Measurement info composed.
Finding samples for g:\MOUS_204\sub-V1001\meg\sub-V1001_task-visual_meg.ds\sub-V1001_task-visual_meg.meg4: 
    System clock channel is available, chec